# TinyCeNN-LM — Rigorous Continuation Distillation

This notebook **continues the existing Transformer-free CeNN student** instead of restarting.

Protocol:
1. download the published `TinyCeNN-LM-Distilled` checkpoint from Hugging Face;
2. rebuild a **fixed 65,536-token held-out benchmark** from the deterministic 1% FineWeb hash split;
3. resume the same 7-step CeNN architecture for **30M additional shuffled training tokens** at a lower learning rate;
4. target **≥90% teacher-gap recovery**;
5. publish a versioned `TinyCeNN-LM-Distilled-v2` checkpoint;
6. download it again from Hugging Face and reproduce the exact benchmark fingerprint + CE within tolerance.


In [ ]:
import subprocess, sys, pathlib, importlib
subprocess.run(["nvidia-smi"], check=False)

REPO_DIR = pathlib.Path("/content/TinyCeNN-LM")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()
import tinycenn_lm
print("TinyCeNN-LM:", tinycenn_lm.__file__)


## Hugging Face login

Add a Hugging Face **write token** to Colab Secrets as `HF_TOKEN`.


In [ ]:
from huggingface_hub import login, HfApi, snapshot_download

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    login()

api = HfApi()
hf_user = api.whoami()["name"]
print("Logged in as:", hf_user)


In [ ]:
# Resume the best 10M-token CeNN-only checkpoint already published by the first notebook.
SOURCE_HF_REPO = f"{hf_user}/TinyCeNN-LM-Distilled"
resume_student = snapshot_download(repo_id=SOURCE_HF_REPO, repo_type="model")
print("Resume checkpoint:", resume_student)


In [ ]:
# Rigorous continuation configuration
ADDITIONAL_TOKENS = 30_000_000
CONTEXT_LENGTH = 256
BATCH_SIZE = 4
GRAD_ACCUM = 4

# Lower LR for continuation from the already-distilled checkpoint.
LEARNING_RATE = 3e-4

CENN_STEPS = 7
DILATIONS = "1,2,4,8,16,32,64"

TEMPERATURE = 2.0
CE_WEIGHT = 1.0
KL_WEIGHT = 1.0
HIDDEN_WEIGHT = 0.25

# Rigorous benchmark settings.
SHUFFLE_BUFFER = 4096
EVAL_BATCHES = 64
EVAL_BATCH_SIZE = 4
EVAL_EVERY = 250
TARGET_GAP_RECOVERY = 0.90

OUTPUT_DIR = str(REPO_DIR / "checkpoints/cenn-student-rigorous-v2")

print("Held-out benchmark tokens:", EVAL_BATCHES * EVAL_BATCH_SIZE * CONTEXT_LENGTH)


In [ ]:
# Continue distillation from the published best checkpoint.
# Clear only this run's output directories; the resume checkpoint lives in the HF cache.
import shutil
for path in (pathlib.Path(OUTPUT_DIR), pathlib.Path(OUTPUT_DIR + "-best")):
    if path.exists():
        shutil.rmtree(path)

cmd = [
    sys.executable, str(REPO_DIR / "scripts/train_distill_rigorous.py"),
    "--resume-student-dir", resume_student,
    "--max-tokens", str(ADDITIONAL_TOKENS),
    "--context-length", str(CONTEXT_LENGTH),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--learning-rate", str(LEARNING_RATE),
    "--steps", str(CENN_STEPS),
    "--dilations", DILATIONS,
    "--temperature", str(TEMPERATURE),
    "--ce-weight", str(CE_WEIGHT),
    "--kl-weight", str(KL_WEIGHT),
    "--hidden-weight", str(HIDDEN_WEIGHT),
    "--shuffle-buffer", str(SHUFFLE_BUFFER),
    "--eval-batches", str(EVAL_BATCHES),
    "--eval-batch-size", str(EVAL_BATCH_SIZE),
    "--eval-every", str(EVAL_EVERY),
    "--target-gap-recovery", str(TARGET_GAP_RECOVERY),
    "--output-dir", OUTPUT_DIR,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
# Inspect the rigorous report.
import json

report_path = pathlib.Path(OUTPUT_DIR) / "distillation_report.json"
report = json.loads(report_path.read_text())

summary = {
    "status": report["status"],
    "benchmark_protocol": report["benchmark_protocol"],
    "held_out_tokens": report["evaluation"]["tokens"],
    "evaluation_fingerprint": report["evaluation"]["fingerprint_sha256"],
    "previous_training_tokens": report["previous_training_tokens"],
    "tokens_this_run": report["seen_tokens_this_run"],
    "cumulative_training_tokens": report["cumulative_training_tokens"],
    "cold_student_ce": report["cold_initial"]["student_ce"],
    "run_start_ce": report["run_start"]["student_ce"],
    "best_student_ce": report["best"]["student_ce"],
    "teacher_ce": report["best"]["teacher_ce"],
    "best_student_ppl": report["best"]["student_ppl"],
    "teacher_ppl": report["best"]["teacher_ppl"],
    "teacher_gap_recovery_percent": 100 * report["teacher_gap_recovery_fraction"],
    "target_reached": report["target_gap_recovery_reached"],
}
print(json.dumps(summary, indent=2))

if report["status"] == "diverged":
    raise RuntimeError("Continuation distillation diverged.")


In [ ]:
# Select the rigorous best checkpoint.
best_dir = pathlib.Path(OUTPUT_DIR + "-best")
publish_dir = best_dir if best_dir.exists() else pathlib.Path(OUTPUT_DIR)
print("Publishing:", publish_dir)

card = f'''---
base_model: arnir0/Tiny-LLM
library_name: transformers
pipeline_tag: text-generation
tags:
- cenn
- knowledge-distillation
- transformer-free
- language-modeling
- recurrent-neural-network
- rigorous-benchmark
---

# TinyCeNN-LM Distilled v2

Transformer-free CeNN student distilled from `arnir0/Tiny-LLM`.

## Architecture
- Transformer layers remaining: 0
- CeNN recurrent steps: {report['cenn_steps']}
- CeNN receptive field: {report['cenn_receptive_field']} tokens
- Trainable CeNN parameters: {report['parameters']['trainable']:,}

## Rigorous benchmark
- Protocol: {report['benchmark_protocol']}
- Deterministic held-out tokens: {report['evaluation']['tokens']:,}
- Benchmark SHA256: `{report['evaluation']['fingerprint_sha256']}`
- Cumulative distillation tokens: {report['cumulative_training_tokens']:,}
- Best student CE: {report['best']['student_ce']:.6f}
- Teacher CE: {report['best']['teacher_ce']:.6f}
- Best student PPL: {report['best']['student_ppl']:.3f}
- Teacher PPL: {report['best']['teacher_ppl']:.3f}
- Teacher-gap recovery: {100*report['teacher_gap_recovery_fraction']:.2f}%

Load via `tinycenn_lm.build_cenn_student()`.
'''
(publish_dir / "README.md").write_text(card, encoding="utf-8")
if publish_dir != report_path.parent:
    (publish_dir / "distillation_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")


In [ ]:
# Publish as a versioned model so the original 10M checkpoint remains reproducible.
HF_MODEL_NAME = "TinyCeNN-LM-Distilled-v2"
HF_REPO_ID = f"{hf_user}/{HF_MODEL_NAME}"

api.create_repo(HF_REPO_ID, repo_type="model", private=False, exist_ok=True)
api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=str(publish_dir),
    commit_message=(
        f"Rigorous CeNN continuation: "
        f"{report['cumulative_training_tokens']:,} cumulative tokens, "
        f"{100*report['teacher_gap_recovery_fraction']:.2f}% gap recovery"
    ),
)
print(f"https://huggingface.co/{HF_REPO_ID}")


In [ ]:
# Re-download the published artifact and reproduce the exact held-out benchmark.
parity_cmd = [
    sys.executable,
    str(REPO_DIR / "scripts/eval_distilled.py"),
    "--hf-repo", HF_REPO_ID,
    "--ce-tolerance", "0.02",
]
subprocess.run(parity_cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
# Side-by-side deterministic generation: teacher vs remotely reloaded CeNN student.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm import build_cenn_student, CeNNReplacementLayer

downloaded_v2 = snapshot_download(repo_id=HF_REPO_ID, repo_type="model")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = (
    torch.bfloat16
    if device.type == "cuda" and torch.cuda.is_bf16_supported()
    else (torch.float16 if device.type == "cuda" else torch.float32)
)

tokenizer = AutoTokenizer.from_pretrained(downloaded_v2)
student = build_cenn_student(downloaded_v2, device=device, dtype=dtype).eval()
teacher = AutoModelForCausalLM.from_pretrained(
    "arnir0/Tiny-LLM",
    dtype=dtype if device.type == "cuda" else None,
    attn_implementation="sdpa",
).to(device).eval()

assert len([m for m in student.modules() if isinstance(m, CeNNReplacementLayer)]) == 1
assert not any("self_attn" in name or "mlp" in name for name, _ in student.named_modules())
print("Transformer-free structure: PASS")

prompts = [
    "The capital of Austria is",
    "Artificial intelligence can help",
    "A small language model",
    "In the future, efficient AI",
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    generation_args = dict(
        max_new_tokens=40,
        do_sample=False,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    with torch.inference_mode():
        student_out = student.generate(**inputs, **generation_args)
        teacher_out = teacher.generate(**inputs, **generation_args)

    print("\nPROMPT:", prompt)
    print("TEACHER:", tokenizer.decode(teacher_out[0], skip_special_tokens=True))
    print("STUDENT:", tokenizer.decode(student_out[0], skip_special_tokens=True))
